# BeautifulSoup Documentation Analytics
**Owner:** Hung (A) — AIO Program · FPT University

This notebook walks through all 10 analytics questions with code and narrative.
Run `python -m pipeline.pipeline` before executing this notebook.

In [ ]:
import sys
sys.path.insert(0, '..')  # make shared/, pipeline/ importable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # pyright: ignore[reportMissingModuleSource]
import plotly.express as px

from shared.utils import load_sections, load_links, load_code_examples
from shared.constants import LINK_TYPES

sections = load_sections()
links    = load_links()
code     = load_code_examples()

print(f'Sections : {len(sections)} rows')
print(f'Links    : {len(links)} rows')
print(f'Code     : {len(code)} rows')

## Dataset Overview

In [ ]:
sections.head()

In [ ]:
sections.describe()

## Q1 — How many sections are in the documentation?

In [ ]:
total = len(sections)
print(f'Total sections: {total}')
print(sections['section_level'].value_counts().sort_index().rename({1:'H1',2:'H2',3:'H3'}))

## Q2 — Which section has the highest word count?

In [ ]:
top = sections.loc[sections['word_count'].idxmax()]
print(f"Section : {top['section_title']}")
print(f"Words   : {top['word_count']}")

fig = px.bar(
    sections.nlargest(10, 'word_count'),
    x='word_count', y='section_title', orientation='h',
    title='Top 10 Sections by Word Count',
    labels={'word_count':'Words','section_title':'Section'}
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## Q3 — Which section contains the most code examples?

In [ ]:
counts = code.groupby('section_title').size().sort_values(ascending=False)
print(f"Section : {counts.index[0]}")
print(f"Count   : {counts.iloc[0]}")
print(counts.head(10))

## Q4 — Which section contains the most links?

In [ ]:
link_counts = links.groupby('section_title').size().sort_values(ascending=False)
print(f"Section : {link_counts.index[0]}")
print(f"Count   : {link_counts.iloc[0]}")

## Q5 — Top 10 most frequent technical keywords

In [ ]:
import re
from collections import Counter
from shared.constants import STOPWORDS

all_text = ' '.join(sections['section_text'].fillna('')).lower()
words    = re.findall(r'\b[a-z]{4,}\b', all_text)
counts   = Counter(w for w in words if w not in STOPWORDS)
top10    = counts.most_common(10)

for rank, (word, count) in enumerate(top10, 1):
    print(f'{rank:2}. {word:<20} {count}')

## Q6 — How many internal and external links exist?

In [ ]:
type_counts = links['link_type'].value_counts()
print(type_counts.to_string())

fig = px.pie(values=type_counts.values, names=type_counts.index,
             title='Link Type Distribution')
fig.show()

## Q7 — How many code examples use find_all()?

In [ ]:
n = int(code['contains_find_all'].sum())
pct = n / len(code) * 100
print(f'find_all() appears in {n} / {len(code)} examples ({pct:.1f}%)')

## Q8 — How many code examples use get_text()?

In [ ]:
n = int(code['contains_get_text'].sum())
pct = n / len(code) * 100
print(f'get_text() appears in {n} / {len(code)} examples ({pct:.1f}%)')

## Q9 (team-proposed) — Average word count per section

In [ ]:
avg = sections['word_count'].mean()
std = sections['word_count'].std()
print(f'Mean   : {avg:.1f} words')
print(f'Median : {sections["word_count"].median():.1f} words')
print(f'Std    : {std:.1f}')

fig = px.histogram(sections, x='word_count', nbins=20,
                   title='Distribution of Section Word Counts')
fig.show()

## Q10 (team-proposed) — Sections with zero code examples

In [ ]:
no_code = sections[sections['code_block_count'] == 0]
print(f'{len(no_code)} sections have no code blocks')
print(no_code[['section_title','section_level','word_count']].to_string(index=False))

## Key Findings

*(Fill in after running above cells)*

- **Q1:** ...
- **Q2:** ...
- **Q5:** ...
- **Q7:** ...


## Limitations

- Requires live internet for initial HTML download.
- Section boundaries depend on h1/h2/h3 heading structure; unnested content is attributed to 'Unknown'.
- Keyword analysis uses simple frequency — no stemming or lemmatisation.


## Conclusion

*(Write after all analysis is complete)*
